# MD sandbox — turn the knobs

A small, fast MD playground: swap the protein, temperature, seed, and (in Section 2) the force field, water model, ensemble, thermostat, and timestep — and watch how the physics responds. All runs are **in memory** (no files). The prep + run machinery lives in `mdtsandbox.py` (which reuses `mdtutorial`), so this notebook stays about the *choices*, not the plumbing.

In [ ]:
# --- environment on-ramp: make sure the MD stack + the modules are importable in THIS kernel ---
import importlib.util, sys, os, subprocess
_missing = [m for m in ("openmm", "pdbfixer", "mdtraj", "py3Dmol") if importlib.util.find_spec(m) is None]
if _missing and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj", "py3Dmol"], check=False)
    _missing = [m for m in _missing if importlib.util.find_spec(m) is None]
if _missing:
    raise SystemExit(f"Missing in this kernel: {_missing}. Select your MD-tutorial conda kernel "
                     "(Kernel > Change Kernel). If it isn't built yet: conda env create -f environment.yml; "
                     "if it exists but is stale: conda env update -f environment.yml.")
_BASE = os.environ.get("MDTUTORIAL_BASE", "https://raw.githubusercontent.com/todd471/MD_tutorial/main")
for _mod in ("mdtutorial.py", "mdtsandbox.py"):          # grab the shipped modules if they aren't alongside
    if not os.path.exists(_mod) and importlib.util.find_spec(_mod[:-3]) is None:
        import urllib.request
        try: urllib.request.urlretrieve(f"{_BASE}/{_mod}", _mod); print("fetched", _mod)
        except Exception as e: print("Place", _mod, "next to this notebook.", e)
import numpy as np, matplotlib.pyplot as plt
import mdtsandbox as sb

In [ ]:
# the tested PDB menu (any RCSB id also works off-menu)
print("available PDBs:")
for k, v in sb.PDB_MENU.items():
    print(f"  {k}  {v}")

## Section 1 — the basic knobs

In [ ]:
# SANDBOX KNOBS (edit me!)
PDB       = "1L2Y"     # a key from sb.PDB_MENU (or any RCSB id)
TEMP_K    = 300        # temperature in kelvin (try 280 vs 360)
SEED      = 2024       # same seed + same GPU model -> identical run; change it for a new trajectory
PROD_PS   = 50         # production length per run, ps (longer = more sampling, slower)
N_REPEATS = 3          # independent runs (different seeds) to see the spread
print(f"config: {PDB} @ {TEMP_K} K, {PROD_PS} ps x {N_REPEATS} repeat(s), base seed {SEED}")
print("       ", sb.PDB_MENU.get(PDB, "(off-menu — fetched from RCSB; some structures need extra cleanup)"))

In [ ]:
# run the repeats and overlay the four generic observables
runs = []; traj_last = None
for r in range(N_REPEATS):
    traj_last, cv = sb.run(PDB, TEMP_K, SEED + r, PROD_PS)
    runs.append(cv)
    print(f"repeat {r} (seed {SEED+r}): RMSD end {cv['rmsd'][-1]:.1f} A | Rg {cv['rg'].mean():.1f} A | helix {cv['helix'].mean():.2f}")

fig, ax = plt.subplots(2, 2, figsize=(11, 6.5), facecolor="white")
panels = [("rmsd", "all-atom RMSD (Å)"), ("rg", "radius of gyration (Å)"),
          ("helix", "helix fraction"), ("ete", "end-to-end Cα (Å)")]
colors = ["#1E90FF", "#FF8C00", "#CC79A7", "#009E73"]
for a, (key, lab) in zip(ax.flat, panels):
    for r, cv in enumerate(runs):
        a.plot(cv["ps"], cv[key], lw=1, color=colors[r % len(colors)], label=f"repeat {r}")
    a.set_xlabel("time (ps)"); a.set_ylabel(lab)
    if key == "helix": a.set_ylim(0, 1)
ax[0, 0].legend(fontsize=8)
fig.suptitle(f"{PDB} @ {TEMP_K} K — {PROD_PS} ps × {N_REPEATS} — generic observables", fontsize=13)
fig.tight_layout(); plt.show()

In [ ]:
# watch the last trajectory (drag to rotate; the animation loops on its own — no play button)
import py3Dmol, tempfile, os
_p = tempfile.mktemp(suffix=".pdb"); traj_last.save_pdb(_p)
view = py3Dmol.view(width=520, height=420)
view.addModelsAsFrames(open(_p).read(), "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.animate({"loop": "forward", "interval": 80}); view.zoomTo()
os.remove(_p); view.show()

### Temperature sweep
Same structure, several temperatures, one observable overlaid.

In [ ]:
TEMPS    = [280, 320, 360]   # kelvin -- 3 well-spread points that bracket TC5b's Tm (~315 K); enough for the trend, quick on a T4
OBS      = "rmsd"                            # "rmsd" | "rg" | "helix" | "ete"
SWEEP_PS = 500                              # ps per temperature
obs_label = {"rmsd": "all-atom RMSD (Å)", "rg": "radius of gyration (Å)",
             "helix": "helix fraction", "ete": "end-to-end Cα (Å)"}[OBS]
plt.figure(figsize=(7.5, 4.2), facecolor="white")
for T, col in zip(TEMPS, plt.cm.turbo(np.linspace(0.1, 0.9, len(TEMPS)))):
    _, cv = sb.run(PDB, T, SEED, SWEEP_PS)
    plt.plot(cv["ps"], cv[OBS], lw=1.4, color=col, label=f"{T} K")
    print(f"{T} K: <{OBS}> = {cv[OBS].mean():.2f}")
plt.xlabel("time (ps)"); plt.ylabel(obs_label)
plt.title(f"{PDB}: {obs_label} vs temperature"); plt.legend(title="T"); plt.tight_layout(); plt.show()

## Section 2 — change the physics
Beyond *which* protein: swap the **force field + water model**, the **ensemble** (NVE/NVT/NPT), the **thermostat**, and the **timestep** (uses `PDB`, `TEMP_K`, `SEED` from Section 1).

### What these knobs mean
- **Force field** — the parameterized potential energy function (bond/angle/torsion + van der Waals + electrostatics) that *is* the physics. AMBER (ff14SB; Maier *et al.* 2015) and CHARMM (CHARMM36; Best *et al.* 2012) are independent parameterizations fit to different reference data; they mostly agree but differ in the details (the comparison below shows this).
- **Water model** — how each water is represented: 3 sites (TIP3P, SPC/E) or 4 (TIP4P-Ew, OPC, with an off-atom charge). Presets: TIP3P (Jorgensen *et al.* 1983), SPC/E (Berendsen *et al.* 1987), TIP4P-Ew (Horn *et al.* 2004), OPC (Izadi *et al.* 2014). It is **co-parameterized with the force field**, so they come as a *matched pair* — which is why they are offered as presets, not free choices. More sites → more faithful, more costly.
- **Ensemble** — which macroscopic quantities are held fixed while the rest fluctuate (grounded in notebook 02): **NVE** (isolated, no thermostat — the total-energy panel should stay ~flat), **NVT** (a thermostat holds temperature; the workhorse, and Section 1's default), **NPT** (a barostat lets the box breathe to hold pressure, so you get the right **density**).
- **Thermostat** (NVT/NPT) — how T is held: **Langevin** adds stochastic friction + random kicks (robust; that noise is exactly what made 'fresh context' matter for reproducibility); **Nosé–Hoover** is a deterministic extended-system thermostat.
- **Timestep** — how far each step advances. **2 fs** is standard *with* `HBonds` constraints (which freeze the fastest X–H vibrations); smaller is more stable but slower, and 4 fs needs hydrogen-mass repartitioning.

The plot shows the **thermodynamic vital signs** — the *same* panels as notebook 02 (potential energy, temperature, total energy, volume/density) — because *those* are what the levers move; the structural observables barely budge on a 60 ps run. Flip **NVE ↔ NVT ↔ NPT** and watch which panel goes flat.

In [ ]:
# PHYSICS KNOBS (edit me!)
print("force-field + water presets:")
for _k in sb.FF_MENU: print("   ", _k)
FORCEFIELD  = "CHARMM36 + TIP3P"   # a key from sb.FF_MENU
ENSEMBLE    = "NPT"                # "NVE" | "NVT" | "NPT" (box breathes)
THERMOSTAT  = "Langevin"          # "Langevin" | "Nose-Hoover"  (NVT/NPT only)
TIMESTEP_FS = 2                    # 1 or 2 fs (with HBonds constraints)
PROD_PS2    = 60                   # NPT wants a little time for the box to settle
_th = "" if ENSEMBLE == "NVE" else f" ({THERMOSTAT})"
print(f"\nphysics: {PDB} @ {TEMP_K} K | {FORCEFIELD} | {ENSEMBLE}{_th} | {TIMESTEP_FS} fs | {PROD_PS2} ps")

In [ ]:
# run with the chosen physics; plot the THERMODYNAMIC vital signs -- the SAME panels as notebook 02.
# THESE respond to the levers; the structural observables (RMSD, Rg, ...) barely move on a 60 ps run.
t2, cv2 = sb.run(PDB, TEMP_K, SEED, PROD_PS2, ff_preset=FORCEFIELD, ensemble=ENSEMBLE,
                 thermostat=THERMOSTAT, timestep_fs=TIMESTEP_FS)
_vkey, _vlab = ("density", "density (g/mL)") if ENSEMBLE == "NPT" else ("volume", "box volume (nm³)")
panels = [("potential_energy", "potential energy (kJ/mol)"), ("temperature", "temperature (K)"),
          ("total_energy", "total energy (kJ/mol)"), (_vkey, _vlab)]
fig, ax = plt.subplots(2, 2, figsize=(11, 6.5), facecolor="white")
for a, (key, lab) in zip(ax.flat, panels):
    a.plot(cv2["ps"], cv2[key], lw=1.0, color="#1E90FF")
    a.axhline(np.mean(cv2[key]), color="k", ls="--", lw=1.0)     # flat MEAN = stable; the scatter is real thermal noise
    a.set_xlabel("time (ps)"); a.set_ylabel(lab)
# what to LOOK for, per ensemble -- the payoff of flipping the lever
note = {"NVE": "NVE — total energy ~flat (conserved by the integrator); temperature wanders (nothing holds it)",
        "NVT": "NVT — temperature held by the thermostat; total energy is NOT flat (exchanged); box volume fixed",
        "NPT": "NPT — temperature held; the box breathes, so density settles toward ~1 g/mL"}.get(ENSEMBLE, "")
_th = "" if ENSEMBLE == "NVE" else f" · {THERMOSTAT}"
fig.suptitle(f"{PDB} · {FORCEFIELD} · {ENSEMBLE}{_th} · {TIMESTEP_FS} fs — thermodynamic vital signs", fontsize=12)
if note: fig.text(0.5, -0.02, note, ha="center", fontsize=9.5, color="0.25")
fig.tight_layout(); plt.show()

### Force-field comparison
Same protein, seed, and ensemble (NVT); two force fields.

In [ ]:
COMPARE = ["amber14 + TIP3P", "CHARMM36 + TIP3P"]   # any two keys from sb.FF_MENU
OBS2    = "rmsd"
_lab = {"rmsd": "all-atom RMSD (Å)", "rg": "radius of gyration (Å)",
        "helix": "helix fraction", "ete": "end-to-end Cα (Å)"}[OBS2]
ff_trajs = {}                                                # keep the trajectories for the side-by-side view below
plt.figure(figsize=(7.5, 4.2), facecolor="white")
for _ffp, _col in zip(COMPARE, ["#1E90FF", "#FF8C00"]):
    _traj, _cvc = sb.run(PDB, TEMP_K, SEED, PROD_PS2, ff_preset=_ffp)   # NVT default
    ff_trajs[_ffp] = _traj
    plt.plot(_cvc["ps"], _cvc[OBS2], lw=1.4, color=_col, label=_ffp)
    print(f"{_ffp:24s}: <{OBS2}> = {_cvc[OBS2].mean():.2f}")
plt.xlabel("time (ps)"); plt.ylabel(_lab)
plt.title(f"{PDB}: {OBS2} under two force fields"); plt.legend(); plt.tight_layout(); plt.show()

### …and side by side — the two trajectories, animated
Drag either molecule (they rotate together); the animation loops on its own. Same protein, same seed — the wiggling is the **force field's** influence. Each is aligned to its own first frame, so you are watching *internal* motion, not tumbling.

In [ ]:
# side-by-side animated trajectories (left/right = the two force fields in COMPARE)
import py3Dmol, tempfile, os
print("left:", COMPARE[0], "  |  right:", COMPARE[1])
view = py3Dmol.view(viewergrid=(1, 2), width=900, height=400, linked=True)
for _col, _ffp in enumerate(COMPARE):
    _t = ff_trajs[_ffp]
    _sub = _t[::max(1, _t.n_frames // 50)]                          # ~50 frames per side
    _tf = tempfile.NamedTemporaryFile(suffix=".pdb", delete=False); _tf.close()
    _sub.save_pdb(_tf.name); _txt = open(_tf.name).read(); os.remove(_tf.name)
    view.addModelsAsFrames(_txt, "pdb", viewer=(0, _col))
    view.setStyle({"cartoon": {"color": "spectrum"}}, viewer=(0, _col))
    view.zoomTo(viewer=(0, _col))
view.animate({"loop": "forward"})                                  # loops automatically -- no play button
view.show()

### Explicit vs implicit solvent
The other major solvent decision (its own item on the prep checklist): model **every water molecule** explicitly, or replace the solvent with a **continuum** — here **GBn2**, a generalized-Born model. Implicit drops the water box entirely (**~10–20× fewer atoms**, so much faster), but solvation becomes an average field: no explicit water structure or water-mediated hydrogen bonds, and folds often **over-compact**. Both runs below use the **same AMBER14 protein force field**, so *only the solvent treatment differs* — and implicit has no box, so it is NVT/NVE only (density is undefined).

In [ ]:
# same protein + AMBER protein FF; ONLY the solvent treatment differs. Implicit (GBn2) has no water box.
import time
SOLV_OBS = "rg"      # a structural observable to compare: "rmsd" | "rg" | "helix" | "ete"
_sl = {"rmsd": "all-atom RMSD (Å)", "rg": "radius of gyration (Å)",
       "helix": "helix fraction", "ete": "end-to-end Cα (Å)"}[SOLV_OBS]
_walls = {}
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), facecolor="white")
for _solv, _col in zip(("explicit", "implicit"), ["#1E90FF", "#FF8C00"]):
    _t0 = time.time()
    _tr, _cv = sb.run(PDB, TEMP_K, SEED, PROD_PS2, ff_preset="amber14 + TIP3P", ensemble="NVT", solvent=_solv)
    _walls[_solv] = time.time() - _t0
    ax[0].plot(_cv["ps"], _cv[SOLV_OBS], lw=1.4, color=_col, label=f"{_solv} ({_tr.n_atoms} atoms)")
    print(f"{_solv:9s}: {_tr.n_atoms:4d} protein atoms | {_walls[_solv]:6.1f}s wall | <{SOLV_OBS}> = {_cv[SOLV_OBS].mean():.2f}")
ax[0].set_xlabel("time (ps)"); ax[0].set_ylabel(_sl); ax[0].legend(fontsize=8)
ax[0].set_title(f"{PDB}: {SOLV_OBS} — explicit vs implicit")
ax[1].bar(list(_walls), list(_walls.values()), color=["#1E90FF", "#FF8C00"])
ax[1].set_ylabel("wall-clock (s)")
ax[1].set_title(f"speed — implicit ≈ {_walls['explicit'] / max(_walls['implicit'], 1e-9):.1f}× faster here")
fig.suptitle("Explicit vs implicit solvent — same protein & AMBER force field, different water treatment", fontsize=12)
fig.tight_layout(); plt.show()

---
### Ideas to try
- **Reproducibility:** same `SEED` on the **same GPU model** → *bit-identical* curves on **CUDA** (seeded + Reference-platform prep and fresh-context dynamics make the whole pipeline deterministic). On OpenCL/CPU, or across a *different* GPU model, expect **statistical, not bitwise** agreement. Change `SEED` for a different trajectory from the same structure. (Full story: `determinism.ipynb`.)
- **Size vs speed:** compare **1UAO** (chignolin, tiny) and **1UBQ** (ubiquitin, large) at the same `PROD_PS`.
- **Fold type:** β-hairpin (**1UAO**) vs 3-helix bundle (**1VII**) vs disulfide-locked (**1CRN**) — watch the RMSD / Rg.
- **Melting:** push `TEMPS` higher on a marginal fold (**1UAO**, **1L2Y**) and watch Rg climb / helix drop.
- **Ensembles (Section 2):** flip **NVE → NVT → NPT** and read the vital-signs panels — is total energy flat under NVE? does density settle near 1 g/mL under NPT? Try 1 fs vs 2 fs, or Langevin vs Nosé–Hoover.
- **Force fields (Section 2):** do amber14 and CHARMM36 agree on your protein's Rg / RMSD?
- **Advanced:** point `PDB` at any small single-chain PDB ID — most will prep, but some need extra cleanup (missing loops, ligands, multiple chains).